# Ukázková úloha z PDF — rozbor

Oficiální zadání okruhu 3 (APR-I-II-3okruhy.pdf). Tohle je jediná úloha, o které víš jistě,
že je reprezentativní. Projdi ji celou, na časovku 60 minut.

**Postup:** přečti si zadání, zavři tenhle notebook a zkus to sám. Až potom se dívej na řešení.

## Zadání

> V následující funkci nalezněte syntaktické i sémantické chyby a opravte je.

Program modifikujte/rozšiřte o následující funkčnost:
- v konstruktoru **zkontrolujte, zda je předaný řetězec platný**
- definujte **vlastnost (property) `stop`**, zda je nutno při daném stavu zastavit
- doplňte metodu, která **porovná (equality) dva semafory**
- **přepínání barev implementujte jako iterátor**

**Výstup:** opravený program + výsledky ladění (co funguje a co nikoliv a proč?)

### Původní kód

**Nejdřív ho spusť tak, jak je** — a pak zkus vytvořit instanci.

In [ ]:
class Semaphore:
    colors = ["red", "yellow", "green"]
    def __init__(color:str):
        self.color = color

        def __str__():
            return self.color

        def nextColor():
            """
             vrací semafor s následující barvou v sekvenci přepínání světel
             """
            return Semaphore(colors[colors.index(self.color)+1])


print("Třída se nadefinovala bez chyby.")
print("metody třídy:", [m for m in dir(Semaphore) if not m.startswith("__")])

**První důležité pozorování:** třída se **nadefinovala bez jediné chybové hlášky**,
a přitom v seznamu metod je jen `colors` — `__str__` ani `nextColor` tam nejsou.

Zkus teď vytvořit instanci:

In [ ]:
try:
    s = Semaphore("red")
except TypeError as e:
    print("TypeError:", e)

Hláška **„takes 1 positional argument but 2 were given"** znamená přesně jedno:
**chybí `self`**. Python předal instanci jako první argument, ale konstruktor počítá
jen s jedním parametrem (`color`, který je ve skutečnosti tou instancí).

**Poučení do zkoušky:** u tříd nestačí kód spustit. Definice projde, i když je uvnitř nesmysl —
musíš **vytvořit instanci a zavolat každou metodu**.

---
## Tvoje řešení

Piš sem. Na řešení se dívej až potom.

In [ ]:
# 1) Oprava

In [ ]:
# 2) Rozšíření — validace + property stop

In [ ]:
# 3) Rozšíření — rovnost + iterátor

In [ ]:
# 4) Testy

---
---
# ŘEŠENÍ — nedívej se, dokud nemáš svoje

## Nalezené chyby

| # | Řádek | Typ | Popis | Oprava |
|---|-------|-----|-------|--------|
| 1 | 3 | sémantická | `def __init__(color: str)` — **chybí `self`**; `color` je ve skutečnosti instance | `def __init__(self, color)` |
| 2 | 6, 9 | sémantická | `__str__` a `nextColor` jsou **odsazené uvnitř `__init__`** — jsou to lokální funkce, ne metody | odsadit o úroveň zpět |
| 3 | 6, 9 | sémantická | i tyhle metody **nemají `self`** | přidat `self` |
| 4 | 13 | sémantická | `colors` bez `self.` → `NameError` uvnitř metody | `self.colors` |
| 5 | 13 | sémantická | `colors.index(...) + 1` **přeteče** na poslední barvě → `IndexError` | `% len(self.colors)` |

**Syntaktická chyba: žádná.** Vnořené funkce jsou legální Python, takže se třída definuje
bez hlášky. Všech pět chyb je sémantických — a tři z nich se projeví až při volání.

### Ukázka chyb 4 a 5 samostatně

In [ ]:
# chyba 4: holý název atributu třídy uvnitř metody
class Ukazka4:
    colors = ["red", "yellow", "green"]
    def spatne(self):
        return colors          # bez self.
    def spravne(self):
        return self.colors

try:
    Ukazka4().spatne()
except NameError as e:
    print("chyba 4 ->", type(e).__name__, ":", e)
print("správně    ->", Ukazka4().spravne())

# chyba 5: přetečení indexu na poslední barvě
colors = ["red", "yellow", "green"]
try:
    print(colors[colors.index("green") + 1])
except IndexError as e:
    print("chyba 5 ->", type(e).__name__, ":", e)
print("s modulem  ->", colors[(colors.index("green") + 1) % len(colors)])

### Opravená verze (jen opravy, bez rozšíření)

In [ ]:
class Semaphore:
    colors = ["red", "yellow", "green"]

    def __init__(self, color: str):          # oprava 1: self
        self.color = color

    def __str__(self):                       # oprava 2 + 3: odsazení a self
        return self.color

    def nextColor(self):
        """Vrací semafor s následující barvou v sekvenci přepínání světel."""
        i = self.colors.index(self.color)    # oprava 4: self.colors
        return Semaphore(self.colors[(i + 1) % len(self.colors)])   # oprava 5: modulo


s = Semaphore("red")
print(s, "->", s.nextColor(), "->", s.nextColor().nextColor())
print("cyklení:", Semaphore("green").nextColor())    # zpátky na red

### Rozšíření — všechny čtyři body ze zadání

In [ ]:
class Semaphore:
    """Semafor se třemi barvami a cyklickým přepínáním."""

    colors = ["red", "yellow", "green"]

    def __init__(self, color: str = "red"):
        # BOD 1: validace v konstruktoru
        if not isinstance(color, str):
            raise TypeError(f"barva musí být řetězec, dostal jsem {type(color).__name__}")
        if color not in self.colors:
            raise ValueError(f"neplatná barva {color!r}, povolené: {self.colors}")
        self.color = color

    # --- textová reprezentace ---
    def __str__(self):
        return self.color

    def __repr__(self):
        return f"Semaphore({self.color!r})"

    # BOD 2: property stop
    @property
    def stop(self) -> bool:
        """True, pokud je při této barvě nutno zastavit."""
        return self.color in ("red", "yellow")

    # BOD 3: rovnost
    def __eq__(self, other):
        if not isinstance(other, Semaphore):
            return NotImplemented          # necháme rozhodnout druhou stranu
        return self.color == other.color

    def __hash__(self):
        return hash(self.color)            # NUTNÉ — __eq__ jinak vypne hashování

    # --- přepínání ---
    def nextColor(self):
        """Vrací NOVÝ semafor s následující barvou."""
        i = self.colors.index(self.color)
        return Semaphore(self.colors[(i + 1) % len(self.colors)])

    # BOD 4: iterátor (generátorem)
    def __iter__(self):
        aktualni = self
        while True:                # nekonečné cyklení — semafor nikdy neskončí
            yield aktualni
            aktualni = aktualni.nextColor()

### Testy

In [ ]:
import itertools

s = Semaphore("red")
print(f"{str(s)=}")                              # 'red'      — __str__
print(f"{repr(s)=}")                             # Semaphore('red')
print(f"{[s]=}")                                 # v seznamu se volá __repr__

print()
print("stop u jednotlivých barev:")
for barva in Semaphore.colors:
    print(f"  {barva:7} -> stop={Semaphore(barva).stop}")

print()
print("přepínání:", " -> ".join(str(x) for x in itertools.islice(Semaphore("red"), 5)))

In [ ]:
# rovnost a hashování
print(f"{Semaphore('red') == Semaphore('red')=}")     # True
print(f"{Semaphore('red') == Semaphore('green')=}")   # False
print(f"{Semaphore('red') == 'red'=}")                # False, ne výjimka
print(f"{len({Semaphore('red'), Semaphore('red'), Semaphore('green')})=}")   # 2

# cyklení přes všechny barvy tam a zpátky
s = Semaphore("red")
for _ in range(len(Semaphore.colors)):
    s = s.nextColor()
print(f"po {len(Semaphore.colors)} krocích zpět na: {s}")

In [ ]:
# chybové stavy
for popis, volani in [
    ("neplatná barva", lambda: Semaphore("modrá")),
    ("není řetězec",   lambda: Semaphore(42)),
    ("prázdný řetězec", lambda: Semaphore("")),
    ("property se závorkami", lambda: Semaphore("red").stop()),
]:
    try:
        volani()
        print(f"{popis:22}: PROŠLO (nemělo!)")
    except (ValueError, TypeError) as e:
        print(f"{popis:22}: {type(e).__name__}: {e}")

### Varianta iterátoru s `__next__`

Kdyby zadání chtělo explicitně `__iter__` + `__next__` místo generátoru:

In [ ]:
class SemaphoreIter:
    """Semafor s explicitním iterátorem — projde barvy PRÁVĚ JEDNOU."""

    colors = ["red", "yellow", "green"]

    def __init__(self, color="red"):
        if color not in self.colors:
            raise ValueError(f"neplatná barva {color!r}")
        self.color = color

    def __str__(self):
        return self.color

    def __iter__(self):
        self._i = self.colors.index(self.color)   # reset stavu při KAŽDÉ iteraci
        self._zbyva = len(self.colors)
        return self                                # jsem sám sobě iterátorem

    def __next__(self):
        if self._zbyva <= 0:
            raise StopIteration                    # POVINNÉ, jinak nekonečný cyklus
        barva = self.colors[self._i % len(self.colors)]
        self._i += 1
        self._zbyva -= 1
        return barva


print(list(SemaphoreIter("yellow")))     # ['yellow', 'green', 'red']

# a proč je generátor lepší: sdílený stav
s = SemaphoreIter("red")
it1, it2 = iter(s), iter(s)
print("dva iterátory nad týmž objektem:", next(it1), next(it2), "  <- sdílejí stav")

g = Semaphore("red")
g1, g2 = iter(g), iter(g)
print("generátor:                       ", next(g1), next(g2), "  <- nezávislé")

---
## Výsledky ladění

*(Tohle je požadovaný výstup ze zadání. U zkoušky vyplň analogicky.)*

### Co funguje
- **Oprava:** `Semaphore("green").nextColor()` vrací `Semaphore('red')` — cyklení přes modulo sedí.
  Původní kód tady padal na `IndexError`.
- **Validace:** `"modrá"` → `ValueError`, `42` → `TypeError`, obojí s vysvětlující hláškou.
- **`property stop`:** `red` a `yellow` → `True`, `green` → `False`. Volá se **bez závorek**;
  se závorkami dá `TypeError: 'bool' object is not callable`.
- **Rovnost:** dva semafory téže barvy jsou si rovné, porovnání s řetězcem vrací `False`
  (ne výjimku — díky `NotImplemented`).
- **Hashování:** `{Semaphore('red'), Semaphore('red'), Semaphore('green')}` má 2 prvky.
  Bez `__hash__` by to spadlo na `TypeError: unhashable type`.
- **Iterátor:** `itertools.islice(Semaphore("red"), 5)` dá `red yellow green red yellow`.
- **Hraniční:** poslední barva v seznamu (`green`) — tam se pozná chybějící modulo.

### Co nefunguje / vědomá omezení
- **Iterátor je nekonečný.** Bez `islice` nebo `break` se cyklus nezastaví. Je to záměr —
  semafor se přepíná pořád dokola. Varianta s `__next__` a `StopIteration` prochází barvy
  právě jednou; obě jsou obhajitelné, záleží na výkladu zadání.
- **Pořadí barev je prostý kruh** podle seznamu ze zadání: `red → yellow → green → red`.
  Reálný semafor jede po zelené na oranžovou a teprve pak na červenou.
  Kdyby to komise chtěla přesně, model by potřeboval seznam `["red", "yellow", "green", "yellow"]`
  nebo směr přepínání.
- **`nextColor` vrací nový objekt** a nemění `self` — plyne z docstringu „vrací semafor".
  Varianta měnící `self.color` je taky legitimní, ale pak nesmí nic vracet.
- **Verze s `__next__` má sdílený stav** — dvě souběžné iterace nad týmž objektem se perou.
  Proto je v hlavním řešení generátor.

### Jak jsem testoval
- Vytvořil instanci, zavolal každou metodu, vypsal objekt samostatně **i v seznamu**
  (kvůli rozdílu `__str__` / `__repr__`).
- Hraniční případy: poslední barva kvůli přetečení indexu, neplatná barva, špatný typ.
- Ověřil vložení do množiny — kontrola, že `__hash__` funguje.
- Ověřil, že se semafor po třech krocích vrátí na výchozí barvu.

### Složitost
- `nextColor` používá `list.index`, tedy $O(n)$ v počtu barev — tady 3, prakticky konstanta.
  Slovníkem barva→index by to bylo $O(1)$, ale u tří prvků je to zbytečná optimalizace.